# Superheterodyne Receiver Chain

Simulation of a basic superheterodyne receiver front-end combining multiple PathSim-RF blocks: an RF amplifier (LNA), a mixer for downconversion, and an IF amplifier. This demonstrates how the blocks compose into a complete signal chain.

## Receiver Architecture

The receiver chain consists of:
1. **LNA** (Low Noise Amplifier): 15 dB gain, IIP3 = +5 dBm
2. **Mixer**: Downconverts RF to IF with 0 dB conversion gain
3. **IF Amplifier**: 20 dB gain, IIP3 = +15 dBm

The RF input at 1000 Hz is downconverted to an IF of 100 Hz using a 900 Hz LO.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Apply PathSim docs matplotlib style
plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope, Spectrum
from pathsim.solvers import RKCK54

from pathsim_rf import RFAmplifier, RFMixer

## System Setup

In [ ]:
# Signal frequencies
f_rf = 1000.0
f_lo = 900.0
f_if = f_rf - f_lo

# Input signal (weak RF signal)
Z0 = 50.0
p_in_dbm = -30.0  # -30 dBm input
p_watts = 10.0 ** (p_in_dbm / 10.0) * 1e-3
v_rf = np.sqrt(2.0 * Z0 * p_watts)

# LO amplitude (strong local oscillator)
v_lo = 1.0  # 1V peak

# Build signal chain
rf_src = Source(func=lambda t: v_rf * np.sin(2 * np.pi * f_rf * t))
lo_src = Source(func=lambda t: v_lo * np.sin(2 * np.pi * f_lo * t))

lna = RFAmplifier(gain=15.0, IIP3=5.0, Z0=Z0)
mixer = RFMixer(conversion_gain=0.0, Z0=Z0)
if_amp = RFAmplifier(gain=20.0, IIP3=15.0, Z0=Z0)

# Observation points at each stage
sco = Scope(labels=['RF input', 'LNA output', 'IF output'])

# Spectrum at output
freq = np.linspace(0, 2500, 500)
spc = Spectrum(freq=freq, labels=['IF output'])

## Connections

The signal flows: RF Source -> LNA -> Mixer (RF port) -> IF Amplifier -> Output. The LO connects to the mixer's LO port.

In [ ]:
connections = [
    Connection(rf_src, lna, sco[0]),
    Connection(lo_src, mixer[1]),
    Connection(lna, mixer[0], sco[1]),
    Connection(mixer, if_amp),
    Connection(if_amp, sco[2], spc),
]

sim = Simulation(
    [rf_src, lo_src, lna, mixer, if_amp, sco, spc],
    connections,
    dt=1 / (20 * (f_rf + f_lo)),
    Solver=RKCK54,
    tolerance_lte_rel=1e-5,
    tolerance_lte_abs=1e-7
)

sim.run(20 / f_if)

## Signal at Each Stage

The scope shows the signal evolving through the receiver chain: the weak RF input, the amplified LNA output, and the downconverted IF output.

In [ ]:
sco.plot()
plt.show()

## Output Spectrum

The IF output spectrum shows the downconverted signal at $f_{\mathrm{IF}} = 100$ Hz along with the image at $f_{\mathrm{RF}} + f_{\mathrm{LO}} = 1900$ Hz.

In [ ]:
spc.plot()
plt.show()